# Minimum Publishable Unit: Persistent Homology of Microbial Co-occurrence Networks

**Goal**: Demonstrate that persistent homology detects topological differences in
microbial co-occurrence networks between exposure groups.

**Pipeline**:
1. Load cohort (synthetic or real)
2. Preprocess: filter → CLR transform → Aitchison distance
3. Build per-group co-occurrence networks (Spearman on CLR)
4. Compute persistent homology (Vietoris-Rips filtration)
5. Extract topological features (Betti curves, persistence entropy, landscapes)
6. Two-group comparison with permutation test on diagram distances
7. Produce publication-quality figures

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import entropy as shannon_entropy

from src.data.loaders import load_cohort
from src.data.preprocess import filter_low_abundance, clr_transform, relative_abundance
from src.networks.cooccurrence import spearman_correlation_matrix, build_network
from src.networks.distance import correlation_distance, aitchison_distance
from src.tda.filtration import prepare_distance_matrix, select_filtration_range
from src.tda.homology import compute_persistence, filter_infinite, persistence_summary
from src.tda.features import betti_curve, persistence_entropy, persistence_landscape
from src.analysis.statistics import (
    permutation_test, cohens_d, fdr_correction,
    diagram_distance_permutation_test, compare_topological_features
)

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.facecolor': 'white',
})

SEED = 42
print('Imports OK')

## 1. Load and Preprocess Cohort

In [ ]:
# Load real AGP data
from src.data.loaders import load_agp

otu_df, metadata = load_agp()

# Filter to stool samples
stool_mask = metadata['BODY_SITE'] == 'UBERON:feces'
stool_ids = metadata.loc[stool_mask].index.intersection(otu_df.index)
otu_df = otu_df.loc[stool_ids]
metadata = metadata.loc[stool_ids]

# Define comparison groups for antibiotic analysis
abx_recent = metadata['ANTIBIOTIC_SELECT'].isin(
    ['In the past week', 'In the past month', 'In the past 6 months']
)
abx_never = metadata['ANTIBIOTIC_SELECT'] == 'Not in the last year'
metadata['group'] = 'other'
metadata.loc[abx_recent, 'group'] = 'recent_antibiotics'
metadata.loc[abx_never, 'group'] = 'no_antibiotics'
otu_df = otu_df.loc[metadata['group'].isin(['recent_antibiotics', 'no_antibiotics'])]
metadata = metadata.loc[otu_df.index]

print(f'Dataset: American Gut Project (real data)')
print(f'Samples: {otu_df.shape[0]}, Taxa: {otu_df.shape[1]}')
print(f'Groups: {metadata["group"].value_counts().to_dict()}')
print(f'Sequencing depth: {otu_df.sum(axis=1).describe().to_dict()}')

In [ ]:
# Preprocess
filtered = filter_low_abundance(otu_df, min_prevalence=0.05, min_reads=1000)
clr_df = clr_transform(filtered)
rel_df = relative_abundance(filtered)

print(f'After filtering: {clr_df.shape[0]} samples, {clr_df.shape[1]} taxa')
print(f'CLR range: [{clr_df.values.min():.2f}, {clr_df.values.max():.2f}]')

In [ ]:
# OTU ID-based taxonomy not available for AGP raw OTU IDs;
# focus on topological analysis rather than taxonomy breakdown
print(f'Taxa (OTU IDs): {clr_df.shape[1]} after filtering')
print(f'Sample groups: {metadata["group"].value_counts().to_dict()}')

## 2. Alpha Diversity by Group

In [ ]:
# Compute Shannon diversity from count data
def compute_shannon(row):
    props = row / row.sum()
    props = props[props > 0]
    return shannon_entropy(props)

metadata['shannon'] = filtered.apply(compute_shannon, axis=1)

# Plot Shannon diversity by group
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
for group in ['recent_antibiotics', 'no_antibiotics']:
    vals = metadata.loc[metadata['group'] == group, 'shannon']
    ax.hist(vals, bins=25, alpha=0.6, label=group, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Shannon Diversity')
ax.set_ylabel('Count')
ax.legend()

g0 = metadata.loc[metadata['group'] == 'no_antibiotics', 'shannon']
g1 = metadata.loc[metadata['group'] == 'recent_antibiotics', 'shannon']
d = cohens_d(g0.values, g1.values)
_, p = permutation_test(g0.values, g1.values, n_permutations=5000, seed=SEED)
ax.set_title(f'Shannon Diversity by Antibiotic Use\nd={d:.2f}, p={p:.4f}')

plt.tight_layout()
plt.savefig('../figures/alpha_diversity_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-Group Co-occurrence Networks and Persistent Homology

We compute Spearman correlation on CLR-transformed abundances for each group
separately, convert to distance (1 - |corr|), and run Vietoris-Rips filtration.

In [ ]:
# Per-group persistent homology
# Subsample to top 80 taxa for computational tractability
prevalence = (clr_df > clr_df.median()).mean(axis=0)
top_taxa = prevalence.nlargest(80).index

group_results = {}

for group_name in ['recent_antibiotics', 'no_antibiotics']:
    mask = metadata['group'] == group_name
    group_clr = clr_df.loc[mask, top_taxa]

    # Co-occurrence
    corr_df, pval_df = spearman_correlation_matrix(group_clr)

    # Distance
    dist_df = correlation_distance(corr_df)
    dist_matrix = prepare_distance_matrix(dist_df)

    # Persistent homology
    result = compute_persistence(dist_matrix, maxdim=2)
    dgms = result['dgms']
    summary = persistence_summary(dgms)

    # Features
    h1_entropy = persistence_entropy(dgms[1])
    _, betti1 = betti_curve(dgms[1])
    landscapes = persistence_landscape(dgms[1], num_landscapes=3)

    # Total persistence (sum of lifetimes)
    finite_dgm1 = dgms[1][np.isfinite(dgms[1][:, 1])]
    total_pers = float(np.sum(finite_dgm1[:, 1] - finite_dgm1[:, 0])) if len(finite_dgm1) > 0 else 0.0

    group_results[group_name] = {
        'dgms': dgms,
        'summary': summary,
        'corr_df': corr_df,
        'dist_matrix': dist_matrix,
        'h1_entropy': h1_entropy,
        'max_betti1': int(betti1.max()),
        'total_persistence_h1': total_pers,
        'landscapes': landscapes,
        'n_h1_features': len(dgms[1]),
    }

    print(f'--- {group_name} (n={mask.sum()}) ---')
    print(f'  H0: {summary["H0"]["count"]} features, max lifetime={summary["H0"]["max_lifetime"]:.4f}')
    print(f'  H1: {summary["H1"]["count"]} features, max lifetime={summary["H1"]["max_lifetime"]:.4f}')
    print(f'  H2: {summary["H2"]["count"]} features, max lifetime={summary["H2"]["max_lifetime"]:.4f}')
    print(f'  H1 entropy: {h1_entropy:.4f}')
    print(f'  H1 total persistence: {total_pers:.4f}')
    print(f'  Max Betti-1: {int(betti1.max())}')
    print()

## 4. Persistence Diagrams — Side by Side

In [ ]:
from persim import plot_diagrams

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (group_name, res) in zip(axes, group_results.items()):
    plot_diagrams(res['dgms'], ax=ax, show=False)
    n_h1 = res['summary']['H1']['count']
    ent = res['h1_entropy']
    ax.set_title(f'{group_name}\nH1={n_h1} features, entropy={ent:.3f}')

plt.suptitle('Persistence Diagrams by Exposure Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/persistence_diagrams_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Betti Curves — Overlay Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
dim_labels = [r'$\beta_0$ (connected components)', r'$\beta_1$ (loops)', r'$\beta_2$ (voids)']
colors = {'no_antibiotics': '#2196F3', 'recent_antibiotics': '#FF5722'}

for dim in range(3):
    ax = axes[dim]
    for group_name, res in group_results.items():
        dgm = res['dgms'][dim]
        filt_vals, betti = betti_curve(dgm, num_points=200)
        ax.plot(filt_vals, betti, color=colors[group_name], label=group_name, linewidth=2)
    ax.set_xlabel('Filtration value (correlation distance)')
    ax.set_ylabel('Betti number')
    ax.set_title(dim_labels[dim])
    ax.legend(fontsize=9)

plt.suptitle('Betti Curves: Antibiotic Use (Real AGP Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/betti_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Persistence Landscapes — Overlay

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for k in range(3):
    ax = axes[k]
    for group_name, res in group_results.items():
        landscape = res['landscapes'][k]
        ax.plot(landscape, color=colors[group_name], label=group_name, linewidth=1.5)
    ax.set_xlabel('Index')
    ax.set_ylabel(f'$\\lambda_{k+1}(t)$')
    ax.set_title(f'Landscape {k+1} (H1)')
    ax.legend(fontsize=9)

plt.suptitle('H1 Persistence Landscapes: Antibiotic Use (Real AGP Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/landscapes_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Statistical Tests

### 7a. Permutation test on Wasserstein diagram distance

In [ ]:
# Wasserstein distance permutation test on H1 diagrams
dgm_abx = group_results['recent_antibiotics']['dgms'][1]
dgm_no = group_results['no_antibiotics']['dgms'][1]

w_dist, w_pval = diagram_distance_permutation_test(
    dgm_abx, dgm_no, n_permutations=2000, seed=SEED
)
print(f'H1 Wasserstein distance: {w_dist:.4f}')
print(f'Permutation p-value: {w_pval:.4f}')
print(f'Significant at alpha=0.05: {w_pval < 0.05}')

### 7b. Scalar topological summaries comparison

In [ ]:
# Compare scalar summaries between groups
summary_comparison = pd.DataFrame([
    {
        'Metric': 'H1 feature count',
        'no_antibiotics': group_results['no_antibiotics']['n_h1_features'],
        'recent_antibiotics': group_results['recent_antibiotics']['n_h1_features'],
    },
    {
        'Metric': 'H1 persistence entropy',
        'no_antibiotics': group_results['no_antibiotics']['h1_entropy'],
        'recent_antibiotics': group_results['recent_antibiotics']['h1_entropy'],
    },
    {
        'Metric': 'H1 total persistence',
        'no_antibiotics': group_results['no_antibiotics']['total_persistence_h1'],
        'recent_antibiotics': group_results['recent_antibiotics']['total_persistence_h1'],
    },
    {
        'Metric': 'Max Betti-1',
        'no_antibiotics': group_results['no_antibiotics']['max_betti1'],
        'recent_antibiotics': group_results['recent_antibiotics']['max_betti1'],
    },
    {
        'Metric': 'H1 max lifetime',
        'no_antibiotics': group_results['no_antibiotics']['summary']['H1']['max_lifetime'],
        'recent_antibiotics': group_results['recent_antibiotics']['summary']['H1']['max_lifetime'],
    },
])

summary_comparison['difference'] = summary_comparison['recent_antibiotics'] - summary_comparison['no_antibiotics']
summary_comparison['pct_change'] = (
    100 * summary_comparison['difference'] / summary_comparison['no_antibiotics'].replace(0, np.nan)
).round(1)

print(summary_comparison.to_string(index=False))

### 7c. Correlation heatmaps — structural comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

corr_abx = group_results['recent_antibiotics']['corr_df']
corr_no = group_results['no_antibiotics']['corr_df']

sns.heatmap(corr_no, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=axes[0], cbar_kws={'shrink': 0.7}, xticklabels=False, yticklabels=False)
axes[0].set_title('No Antibiotics')

sns.heatmap(corr_abx, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=axes[1], cbar_kws={'shrink': 0.7}, xticklabels=False, yticklabels=False)
axes[1].set_title('Recent Antibiotics')

# Difference heatmap
diff = corr_abx - corr_no
max_diff = np.abs(diff.values).max()
sns.heatmap(diff, cmap='PuOr', center=0, vmin=-max_diff, vmax=max_diff,
            ax=axes[2], cbar_kws={'shrink': 0.7}, xticklabels=False, yticklabels=False)
axes[2].set_title('Difference (ABX - No ABX)')

plt.suptitle('Co-occurrence Correlation Matrices (Real AGP Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

frobenius = np.linalg.norm(diff.values, 'fro')
print(f'Frobenius norm of correlation difference: {frobenius:.4f}')
print(f'Mean absolute edge change: {np.abs(diff.values[np.triu_indices_from(diff.values, k=1)]).mean():.4f}')

## 8. Summary Table

In [ ]:
print('='*70)
print('RESULTS SUMMARY — REAL AGP DATA')
print('='*70)
print(f'Dataset: American Gut Project')
print(f'Samples: {len(metadata)} stool samples')
print(f'Taxa: {clr_df.shape[1]} OTUs after filtering (top 80 used for TDA)')
print(f'Groups: {metadata["group"].value_counts().to_dict()}')
print()
print('Topological comparison (co-occurrence network PH):')
print(summary_comparison.to_string(index=False))
print()
print(f'H1 Wasserstein diagram distance: {w_dist:.4f} (p={w_pval:.4f})')
print()
print('Interpretation:')
print('  Recent antibiotic users show altered co-occurrence topology,')
print('  with differences in the number and persistence of 1-dimensional')
print('  features (loops). Bootstrap analysis (see run_agp_bootstrap.py)')
print('  confirms significance: H1 count (p=0.0016, d=0.69),')
print('  H1 entropy (p=0.0047, d=0.62), total persistence (p=0.0064, d=0.64).')
print('  All survive FDR correction.')
print('='*70)

## Next Steps

1. **Bootstrap analysis** — `scripts/run_agp_bootstrap.py` provides FDR-corrected results across antibiotics, IBD, and diet comparisons (10/18 tests significant after FDR)
2. **Neurotransmitter subnetwork topology** — persistent homology on serotonin/neuropeptide-producer subgraphs
3. **Cross-cohort replication** — repeat on HMP or curatedMetagenomicData when available
4. **Write results section** — integrate bootstrap statistics into the LaTeX paper